# Dummy

In [ ]:
data = [
  "He boy",
  "She girl"
]

In [ ]:
VOCAB_SIZE = 4
CONTEXT_LEN = 1
BATCH_SIZE = 2
EPOCHS = 100

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

In [ ]:
_ = torch.manual_seed(123)

In [ ]:
text_data = data

In [ ]:
example_data = text_data[0]
print(example_data)

In [ ]:
vocab = set()

for text in text_data:
    for word in text.split():
        vocab.add(word)

vocab = sorted(vocab)

print(len(vocab))

In [ ]:
# Display token:word mappings
for i, word in enumerate(vocab):
    print(f"{i}: {word}")

In [ ]:
# build word -> index mapping
word_to_id = {word: idx for idx, word in enumerate(vocab)}

# build id -> word mapping
id_to_word = {idx: word for idx, word in enumerate(vocab)}

In [ ]:
def text_to_token_ids(text, word_to_id):
    tokens = text.split()
    return [word_to_id[t] for t in tokens]

def token_ids_to_text(token_ids, id_to_word):
    words = [id_to_word[id] for id in token_ids]
    return " ".join(words)

In [ ]:
print(example_data)

In [ ]:
text_to_token_ids(example_data, word_to_id)

In [ ]:
text_to_token_ids(example_data, word_to_id)[:CONTEXT_LEN]

In [ ]:
text_to_token_ids(example_data, word_to_id)[1:CONTEXT_LEN + 1]

In [ ]:
class LLMDataset(Dataset):
    def __init__(self, texts, word_to_id, max_len):
        self.texts = texts
        self.word_to_id = word_to_id
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = torch.tensor(text_to_token_ids(self.texts[idx], self.word_to_id)[:self.max_len + 1])
        x = tokens[:-1]
        y = tokens[1:]
        return x, y

# create dataset + dataloader
train_dataset = LLMDataset(
    text_data,
    word_to_id,
    CONTEXT_LEN
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(len(train_dataloader))

In [ ]:
example_input_output = next(iter(train_dataloader))

In [ ]:
print(example_input_output)

In [ ]:
import re
formatted = [re.sub(r'\s+', ' ', str(t)) for t in example_input_output]
print("[" + ",\n ".join(formatted) + "]")

In [ ]:
# Example Input
example_input_output[0][0]

In [ ]:
# Example Output (shifted by one token)
example_input_output[1][0]

In [ ]:
class GPTModel(nn.Module):
    def __init__(self):

        super().__init__()
        self.w = nn.Parameter(torch.zeros(1))
        self.b = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return x.float() * self.w + self.b

In [ ]:
model = GPTModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {trainable_params:,}")

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
param_count = defaultdict(int)

for name, param in model.named_parameters():
    top_level = name.split('.')[0]
    param_count[top_level] += param.numel()

for k, v in param_count.items():
    print(f"{k:20s}: {v:,}")

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):

        y_predicted = model(X)
        loss = loss_fn(y_predicted, y.float())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(f"batch: {batch + 1} loss: {loss:>7f}")

In [ ]:
for epoch in range(EPOCHS):
    print("-------------------------------")
    print(f"Epoch: {epoch+1}")
    train(train_dataloader, model, loss_fn, optimizer)
    print("-------------------------------")
print("Done!")

In [ ]:
_ = model.eval()

In [ ]:
def generate(start_text):
    token_id = word_to_id[start_text]
    x = torch.tensor([token_id])
    with torch.no_grad():
        y_predicted = model(x)
    next_token_id = round(y_predicted.item())
    return start_text + " " + id_to_word[next_token_id]

In [ ]:
text = generate("He")
print("Generated text:", text)

In [ ]:
text = generate("She")
print("Generated text:", text)